# Pretrain `base_100m` on Colab

Before running:
1. **Runtime → Change runtime type → A100 GPU** (L4 also works, slower; T4 falls back to fp16).
2. **Secrets** (key icon in the left sidebar) → add `HF_TOKEN` = a Hugging Face *Read* token, and enable notebook access.
   It's needed to download the private dataset.
3. Keep this tab open and the computer awake: Colab Pro stops the runtime when the tab closes.

Checkpoints go to Google Drive, so a disconnect loses at most ~20 minutes: reconnect, run cells 1–4, then the **Resume** cell.

## 1. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
DTYPE = "bf16" if torch.cuda.is_bf16_supported() else "fp16"
print("dtype:", DTYPE)

## 2. Google Drive (checkpoints and logs survive disconnects)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
RUN_ROOT = "/content/drive/MyDrive/transformer-from-scratch"

## 3. Code

In [ ]:
import os
REPO = "/content/transformer-pytorch"
if not os.path.exists(REPO):
    !git clone -q https://github.com/efe-tatar/transformer-pytorch.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!git log --oneline -1
!pip install -q regex wandb tiktoken

## 4. Data (6 GB from the Hugging Face Hub, a few minutes)

In [ ]:
from google.colab import userdata
from huggingface_hub import snapshot_download
snapshot_download("Efe-Tatar/fineweb-edu-3b-bpe32k", repo_type="dataset",
                  local_dir="data/fineweb_edu_3b", token=userdata.get("HF_TOKEN"))
!ls data/fineweb_edu_3b | head -3 && du -sh data/fineweb_edu_3b

## 5. Optional: Weights & Biases

Live loss curves you can check from your phone. Needs a free account at wandb.ai; `wandb.login()` asks for your API key.
Skip this cell to train without it (logs still go to `metrics.jsonl` on Drive).

In [ ]:
import wandb
wandb.login()
USE_WANDB = True

## 6. Smoke test (~5–10 min)

Runs 20 steps, stops with a checkpoint on Drive, then resumes for 20 more. Checks that it doesn't run out of memory, that `torch.compile`
works, that checkpoints write to Drive and that resume works, and measures **tokens/s**. The first steps are slow while
`torch.compile` compiles (1–2 min); look at the later lines.

If it runs out of memory: add `--micro_batch_size=8` to `COMMON` (gradient accumulation doubles automatically).

In [ ]:
COMMON = f"--preset base_100m --attn_impl=sdpa --dtype={DTYPE}"
SMOKE = f"--out_dir={RUN_ROOT}/smoke --max_steps=40 --warmup_steps=10 --log_interval=5 --eval_interval=20 --eval_iters=5 --checkpoint_interval=1000"
!rm -rf {RUN_ROOT}/smoke
!python train.py {COMMON} {SMOKE} --stop_step=20
!python train.py {COMMON} {SMOKE} --resume

## 7. The real run

`MAX_STEPS = 5000` × 0.5M tokens = 2.6B tokens. Use the smoke test's tokens/s to check it fits in your compute units:
hours ≈ 2.6e9 / (tokens/s) / 3600. If it doesn't fit, lower `MAX_STEPS` (e.g. 3000 = 1.6B tokens) **before** starting.
Changing it after starting would change the learning-rate schedule.

In [ ]:
MAX_STEPS = 5000
WANDB = "--wandb=true --run_name=base_100m" if globals().get("USE_WANDB") else ""
RUN = f"--out_dir={RUN_ROOT}/base_100m --max_steps={MAX_STEPS} {WANDB}"
!python train.py {COMMON} {RUN}

## Resume after a disconnect

Run cells 1–4 (and 5 if you use W&B), then this one. Settings must be the same as the original run.

In [ ]:
COMMON = f"--preset base_100m --attn_impl=sdpa --dtype={DTYPE}"
MAX_STEPS = 5000
WANDB = "--wandb=true --run_name=base_100m" if globals().get("USE_WANDB") else ""
RUN = f"--out_dir={RUN_ROOT}/base_100m --max_steps={MAX_STEPS} {WANDB}"
!python train.py {COMMON} {RUN} --resume